# Memakai Convnet Pralatih: Ekstraksi Fitur dan Fine-Tuning

**Buku:** *Deep Learning dengan Python* — Bab 5, Deep Learning untuk Visi Komputer

Memanfaatkan basis konvolusi VGG16 yang dilatih pada ImageNet untuk tugas kucing vs anjing dengan data sedikit.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- **Bug prapemrosesan asli diperbaiki:** VGG16 dilatih dengan `vgg16.preprocess_input` (BGR, pengurangan rerata), bukan skala 1/255.
- `fit_generator`/`evaluate_generator` → `fit`/`evaluate` pada `tf.data`.
- Pembekuan lapisan untuk fine-tuning memakai irisan `conv_base.layers[:-4]`.
- Model terbaik dipilih dengan `ModelCheckpoint` (format `.keras`).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

Notebook ini memakai subset data dari `bab05_02`. Sel berikut akan mengunduh ulang jika belum ada.

In [ ]:
import os, zipfile, shutil, pathlib

URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
zip_path = keras.utils.get_file("kagglecatsanddogs_5340.zip", origin=URL)

asli = pathlib.Path("data/PetImages")
if not asli.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("data")

def berkas_valid(folder):
    """Kembalikan berkas JPEG yang sah (dataset asli memuat beberapa berkas rusak)."""
    hasil = []
    for p in sorted(folder.glob("*.jpg"), key=lambda q: int(q.stem) if q.stem.isdigit() else -1):
        with open(p, "rb") as f:
            if b"JFIF" in f.read(10):
                hasil.append(p)
    return hasil

kecil = pathlib.Path("data/kucing_vs_anjing_kecil")
PEMBAGIAN = {"latih": (0, 1000), "validasi": (1000, 1500), "uji": (1500, 2000)}
if not kecil.exists():
    for kelas_asli, kelas in [("Cat", "kucing"), ("Dog", "anjing")]:
        daftar = berkas_valid(asli / kelas_asli)
        for bagian, (awal, akhir) in PEMBAGIAN.items():
            tujuan = kecil / bagian / kelas
            tujuan.mkdir(parents=True, exist_ok=True)
            for p in daftar[awal:akhir]:
                shutil.copyfile(p, tujuan / p.name)

for bagian in PEMBAGIAN:
    print(bagian, {k: len(list((kecil / bagian / k).glob("*.jpg"))) for k in ["kucing", "anjing"]})

In [ ]:
UKURAN = (150, 150)
ds_latih = keras.utils.image_dataset_from_directory(
    kecil / "latih", image_size=UKURAN, batch_size=32, seed=42)
ds_val = keras.utils.image_dataset_from_directory(
    kecil / "validasi", image_size=UKURAN, batch_size=32, shuffle=False)
ds_uji = keras.utils.image_dataset_from_directory(
    kecil / "uji", image_size=UKURAN, batch_size=32, shuffle=False)
print("Nama kelas:", ds_latih.class_names)   # urutan alfabet: anjing=0, kucing=1

In [ ]:
from keras import layers
from keras.applications import vgg16

conv_base = vgg16.VGG16(weights="imagenet", include_top=False, input_shape=UKURAN + (3,))
conv_base.summary()

## 1. Ekstraksi fitur cepat (tanpa augmentasi)
Jalankan basis konvolusi **sekali** untuk setiap citra, simpan fiturnya, lalu latih pengklasifikasi kecil.

In [ ]:
import numpy as np

def ambil_fitur(dataset):
    semua_fitur, semua_label = [], []
    for citra, label in dataset:
        x = vgg16.preprocess_input(keras.ops.convert_to_numpy(citra))
        semua_fitur.append(conv_base.predict(x, verbose=0))
        semua_label.append(keras.ops.convert_to_numpy(label))
    return np.concatenate(semua_fitur), np.concatenate(semua_label)

f_latih, l_latih = ambil_fitur(ds_latih)
f_val, l_val = ambil_fitur(ds_val)
f_uji, l_uji = ambil_fitur(ds_uji)
print(f_latih.shape)   # (2000, 4, 4, 512)

In [ ]:
import matplotlib.pyplot as plt

def plot_riwayat(riwayat, judul=""):
    h = riwayat.history
    epoch = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epoch, h["accuracy"], "o", ms=3, label="Latih")
    ax[0].plot(epoch, h["val_accuracy"], "-", label="Validasi")
    ax[0].set(xlabel="Epoch", ylabel="Akurasi", title=judul)
    ax[1].plot(epoch, h["loss"], "o", ms=3, label="Latih")
    ax[1].plot(epoch, h["val_loss"], "-", label="Validasi")
    ax[1].set(xlabel="Epoch", ylabel="Loss", title=judul)
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

masukan = keras.Input(shape=f_latih.shape[1:])
x = layers.Flatten()(masukan)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
keluaran = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(masukan, keluaran)
model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])
cb = [keras.callbacks.ModelCheckpoint("ekstraksi_fitur.keras", save_best_only=True, monitor="val_loss")]
riwayat = model.fit(f_latih, l_latih, epochs=20, batch_size=32,
                    validation_data=(f_val, l_val), callbacks=cb)
plot_riwayat(riwayat, "Ekstraksi fitur cepat")

# Untuk verifikasi angka buku (Tabel 5.x)
print(f"Ekstraksi cepat: val acc terbaik {max(riwayat.history['val_accuracy']):.2%} | "
      f"akurasi uji {model.evaluate(f_uji, l_uji, verbose=0)[1]:.2%}")

## 2. Ekstraksi fitur end-to-end dengan augmentasi
Basis konvolusi **dibekukan** agar representasinya tidak rusak oleh gradien besar dari lapisan baru.

In [ ]:
conv_base.trainable = False
augmentasi = keras.Sequential([layers.RandomFlip("horizontal"),
                               layers.RandomRotation(0.1),
                               layers.RandomZoom(0.2)])

masukan = keras.Input(shape=UKURAN + (3,))
x = augmentasi(masukan)
x = vgg16.preprocess_input(x)
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
keluaran = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(masukan, keluaran)
print("Bobot yang dapat dilatih:", len(model.trainable_weights))   # hanya 4 (dua lapisan Dense)

model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])
cb = [keras.callbacks.ModelCheckpoint("ekstraksi_fitur_augmentasi.keras",
                                      save_best_only=True, monitor="val_loss")]
riwayat = model.fit(ds_latih, epochs=50, validation_data=ds_val, callbacks=cb)
plot_riwayat(riwayat, "Ekstraksi fitur + augmentasi")

# Untuk verifikasi angka buku (Tabel 5.x)
print(f"Ekstraksi + augmentasi: val acc terbaik {max(riwayat.history['val_accuracy']):.2%} | "
      f"akurasi uji {model.evaluate(ds_uji, verbose=0)[1]:.2%}")

## 3. Fine-tuning
Buka pembekuan **hanya** empat lapisan teratas (blok 5), lalu latih dengan laju pembelajaran sangat kecil.

In [ ]:
conv_base.trainable = True
for lapisan in conv_base.layers[:-4]:
    lapisan.trainable = False

model.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
              metrics=["accuracy"])          # kompilasi ulang wajib setelah mengubah trainable
cb = [keras.callbacks.ModelCheckpoint("fine_tuning.keras", save_best_only=True, monitor="val_loss")]
riwayat = model.fit(ds_latih, epochs=30, validation_data=ds_val, callbacks=cb)
plot_riwayat(riwayat, "Fine-tuning blok 5")

## 4. Evaluasi akhir pada data uji

In [ ]:
terbaik = keras.models.load_model("fine_tuning.keras")
_, akurasi = terbaik.evaluate(ds_uji, verbose=0)
print(f"Akurasi uji setelah fine-tuning: {akurasi:.2%}")